In [5]:
import numpy as np
import pandas as pd
import os
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# --- Load Data ---
df = pd.read_csv(r"D:\ml_lern\data\kc_house_data.csv")

# Drop unwanted columns
for col in ['id', 'date']:
    if col in df.columns:
        df.drop(col, axis=1, inplace=True)

X = df.drop('price', axis=1).values
y = df['price'].values

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Polynomial features (degree 2)
poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

# Create model save directory
model_dir = r"D:\ml_lern\model"
os.makedirs(model_dir, exist_ok=True)

def evaluate_and_save(model, name, X_tr, X_te, save_name):
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    cv_r2 = cross_val_score(model, X, y, cv=5, scoring='r2').mean()

    print(f" Model: {name}")
    print(f"R² Score: {r2:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"Cross-Validated R² (5-fold): {cv_r2:.4f}")

    # Save model
    path = os.path.join(model_dir, save_name)
    joblib.dump(model, path)
    print(f"✅ Saved {name} model at: {path}")

# Run & save all models:

# Linear family
evaluate_and_save(LinearRegression(), "Simple Linear Regression", X_train_scaled, X_test_scaled, "linear_regression.pkl")
evaluate_and_save(LinearRegression(), "Multiple Linear Regression", X_train_scaled, X_test_scaled, "multiple_linear_regression.pkl")
evaluate_and_save(LinearRegression(), "Polynomial Regression (degree=2)", X_train_poly, X_test_poly, "polynomial_regression.pkl")

# Regularized
evaluate_and_save(Ridge(alpha=1.0), "Ridge Regression", X_train_scaled, X_test_scaled, "ridge_regression.pkl")
evaluate_and_save(Lasso(alpha=0.1), "Lasso Regression", X_train_scaled, X_test_scaled, "lasso_regression.pkl")
evaluate_and_save(ElasticNet(alpha=0.1, l1_ratio=0.5), "Elastic Net", X_train_scaled, X_test_scaled, "elastic_net.pkl")

# Advanced
evaluate_and_save(SVR(kernel='rbf', C=100, epsilon=0.1), "Support Vector Regression", X_train_scaled, X_test_scaled, "svr.pkl")
evaluate_and_save(DecisionTreeRegressor(max_depth=6), "Decision Tree Regressor", X_train, X_test, "decision_tree.pkl")
evaluate_and_save(RandomForestRegressor(n_estimators=100, max_depth=6), "Random Forest Regressor", X_train, X_test, "random_forest.pkl")
evaluate_and_save(XGBRegressor(n_estimators=100, learning_rate=0.1, verbosity=0), "XGBoost Regressor", X_train, X_test, "xgboost.pkl")
evaluate_and_save(LGBMRegressor(n_estimators=100, learning_rate=0.1), "LightGBM Regressor", X_train, X_test, "lightgbm.pkl")
evaluate_and_save(CatBoostRegressor(verbose=0), "CatBoost Regressor", X_train, X_test, "catboost.pkl")
evaluate_and_save(KNeighborsRegressor(n_neighbors=5), "KNN Regressor", X_train_scaled, X_test_scaled, "knn.pkl")

# Save scaler and polynomial features
joblib.dump(scaler, os.path.join(model_dir, "scalers.pkl"))
print(f"✅ Saved scaler at: {os.path.join(model_dir, 'scalers.pkl')}")

joblib.dump(poly, os.path.join(model_dir, "polys.pkl"))
print(f"✅ Saved polynomial features transformer at: {os.path.join(model_dir, 'polys.pkl')}")



📌 Model: Simple Linear Regression
R² Score: 0.7012
MAE: 127493.3421
MSE: 45173046132.7901
RMSE: 212539.5166
Cross-Validated R² (5-fold): 0.6946
✅ Saved Simple Linear Regression model at: D:\ml_lern\model\linear_regression.pkl

📌 Model: Multiple Linear Regression
R² Score: 0.7012
MAE: 127493.3421
MSE: 45173046132.7901
RMSE: 212539.5166
Cross-Validated R² (5-fold): 0.6946
✅ Saved Multiple Linear Regression model at: D:\ml_lern\model\multiple_linear_regression.pkl

📌 Model: Polynomial Regression (degree=2)
R² Score: 0.7987
MAE: 104218.9277
MSE: 30429723841.0830
RMSE: 174441.1759
Cross-Validated R² (5-fold): 0.6946
✅ Saved Polynomial Regression (degree=2) model at: D:\ml_lern\model\polynomial_regression.pkl

📌 Model: Ridge Regression
R² Score: 0.7012
MAE: 127491.4005
MSE: 45173305231.0990
RMSE: 212540.1262
Cross-Validated R² (5-fold): 0.6946
✅ Saved Ridge Regression model at: D:\ml_lern\model\ridge_regression.pkl


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.059e+13, tolerance: 2.259e+11
  model = cd_fast.enet_coordinate_descent(
c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.096e+14, tolerance: 2.258e+11
  model = cd_fast.enet_coordinate_descent(
c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale 


📌 Model: Lasso Regression
R² Score: 0.7012
MAE: 127493.3413
MSE: 45173050018.2845
RMSE: 212539.5258
Cross-Validated R² (5-fold): 0.6946
✅ Saved Lasso Regression model at: D:\ml_lern\model\lasso_regression.pkl


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.999e+14, tolerance: 2.258e+11
  model = cd_fast.enet_coordinate_descent(
c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.108e+14, tolerance: 2.306e+11
  model = cd_fast.enet_coordinate_descent(
c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale 


📌 Model: Elastic Net
R² Score: 0.6989
MAE: 126285.8070
MSE: 45513632130.6280
RMSE: 213339.2419
Cross-Validated R² (5-fold): 0.6548
✅ Saved Elastic Net model at: D:\ml_lern\model\elastic_net.pkl

📌 Model: Support Vector Regression
R² Score: 0.0943
MAE: 191632.0226
MSE: 136924285186.1852
RMSE: 370032.8164
Cross-Validated R² (5-fold): -0.0547
✅ Saved Support Vector Regression model at: D:\ml_lern\model\svr.pkl

📌 Model: Decision Tree Regressor
R² Score: 0.6950
MAE: 112419.2278
MSE: 46113115529.8900
RMSE: 214739.6459
Cross-Validated R² (5-fold): 0.7443
✅ Saved Decision Tree Regressor model at: D:\ml_lern\model\decision_tree.pkl

📌 Model: Random Forest Regressor
R² Score: 0.7833
MAE: 100701.8589
MSE: 32764020498.1302
RMSE: 181008.3437
Cross-Validated R² (5-fold): 0.8040
✅ Saved Random Forest Regressor model at: D:\ml_lern\model\random_forest.pkl

📌 Model: XGBoost Regressor
R² Score: 0.8762
MAE: 70312.4797
MSE: 18718803891.2431
RMSE: 136816.6799
Cross-Validated R² (5-fold): 0.8865
✅ Saved X

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000694 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2337
[LightGBM] [Info] Number of data points in the train set: 17290, number of used features: 18
[LightGBM] [Info] Start training from score 543766.086755
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000841 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2336
[LightGBM] [Info] Number of data points in the train set: 17291, number of used features: 18
[LightGBM] [Info] Start training from score 540558.791510
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000629 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2333
[LightGBM] [Info] Number of data points in the train set: 17291, number of used features: 18
[LightGBM] [Info]

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



📌 Model: LightGBM Regressor
R² Score: 0.8758
MAE: 70379.4031
MSE: 18779348357.0616
RMSE: 137037.7625
Cross-Validated R² (5-fold): 0.8866
✅ Saved LightGBM Regressor model at: D:\ml_lern\model\lightgbm.pkl

📌 Model: CatBoost Regressor
R² Score: 0.9019
MAE: 65850.6796
MSE: 14829023617.8613
RMSE: 121774.4785
Cross-Validated R² (5-fold): 0.9028
✅ Saved CatBoost Regressor model at: D:\ml_lern\model\catboost.pkl

📌 Model: KNN Regressor
R² Score: 0.7799
MAE: 93150.9840
MSE: 33281052532.2385
RMSE: 182430.9528
Cross-Validated R² (5-fold): 0.4932
✅ Saved KNN Regressor model at: D:\ml_lern\model\knn.pkl
✅ Saved scaler at: D:\ml_lern\model\scalers.pkl
✅ Saved polynomial features transformer at: D:\ml_lern\model\polys.pkl


In [ ]:
#  Imports
import numpy as np
import pandas as pd
import os
import joblib
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# --- Load Data ---
df = pd.read_csv(r"D:\ml_lern\data\kc_house_data.csv")

# Drop unwanted columns
for col in ['id', 'date']:
    if col in df.columns:
        df.drop(col, axis=1, inplace=True)

X = df.drop('price', axis=1).values
y = df['price'].values

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Polynomial features (degree 2)
poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

# Create model save directory
model_dir = r"D:\ml_lern\model"
os.makedirs(model_dir, exist_ok=True)

def evaluate_and_save(model, name, X_tr, X_te, save_name):
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    cv_r2 = cross_val_score(model, X, y, cv=5, scoring='r2').mean()

    print(f" Model: {name}")
    print(f"R² Score: {r2:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"Cross-Validated R² (5-fold): {cv_r2:.4f}")

    # Save model
    path = os.path.join(model_dir, save_name)
    joblib.dump(model, path)
    print(f" Saved {name} model at: {path}")

# Run & save all models:

# Linear family
evaluate_and_save(LinearRegression(), "Simple Linear Regression", X_train_scaled, X_test_scaled, "linear_regression.pkl")
evaluate_and_save(LinearRegression(), "Multiple Linear Regression", X_train_scaled, X_test_scaled, "multiple_linear_regression.pkl")
evaluate_and_save(LinearRegression(), "Polynomial Regression (degree=2)", X_train_poly, X_test_poly, "polynomial_regression.pkl")

# Regularized
evaluate_and_save(Ridge(alpha=1.0), "Ridge Regression", X_train_scaled, X_test_scaled, "ridge_regression.pkl")
evaluate_and_save(Lasso(alpha=0.1), "Lasso Regression", X_train_scaled, X_test_scaled, "lasso_regression.pkl")
evaluate_and_save(ElasticNet(alpha=0.1, l1_ratio=0.5), "Elastic Net", X_train_scaled, X_test_scaled, "elastic_net.pkl")

# Advanced
evaluate_and_save(SVR(kernel='rbf', C=100, epsilon=0.1), "Support Vector Regression", X_train_scaled, X_test_scaled, "svr.pkl")
evaluate_and_save(DecisionTreeRegressor(max_depth=6), "Decision Tree Regressor", X_train, X_test, "decision_tree.pkl")
evaluate_and_save(RandomForestRegressor(n_estimators=100, max_depth=6), "Random Forest Regressor", X_train, X_test, "random_forest.pkl")
evaluate_and_save(XGBRegressor(n_estimators=100, learning_rate=0.1, verbosity=0), "XGBoost Regressor", X_train, X_test, "xgboost.pkl")
evaluate_and_save(LGBMRegressor(n_estimators=100, learning_rate=0.1), "LightGBM Regressor", X_train, X_test, "lightgbm.pkl")
evaluate_and_save(CatBoostRegressor(verbose=0), "CatBoost Regressor", X_train, X_test, "catboost.pkl")
evaluate_and_save(KNeighborsRegressor(n_neighbors=5), "KNN Regressor", X_train_scaled, X_test_scaled, "knn.pkl")

# Save scaler and polynomial features
joblib.dump(scaler, os.path.join(model_dir, "scalers.pkl"))
print(f" Saved scaler at: {os.path.join(model_dir, 'scalers.pkl')}")

joblib.dump(poly, os.path.join(model_dir, "polys.pkl"))
print(f" Saved polynomial features transformer at: {os.path.join(model_dir, 'polys.pkl')}")
